# Tactical GNN — 실제 AI Hub 데이터 학습

AI Hub `track1.json` (선수 좌표) + `strategy.json` (전술 라벨) → 실제 데이터 학습

In [ ]:
import torch
TORCH = torch.__version__.split('+')[0]
if torch.version.cuda:
    CUDA = torch.version.cuda.replace('.', '')
    !pip install -q torch-geometric
    !pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
else:
    print(f"CUDA 없음 (CPU/MPS 모드). PyTorch {TORCH}")
    !pip install -q torch-geometric torch-scatter torch-sparse

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, gc, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROJECT_DIR = Path('/content/drive/MyDrive/tactical-gnn')
AIHUB_DIR = PROJECT_DIR / 'data' / 'aihub' / 'training_labels' / '1.동영상분석'
MODEL_DIR = PROJECT_DIR / 'models'
GRAPH_DIR = PROJECT_DIR / 'data' / 'graphs'
for d in [MODEL_DIR, GRAPH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# === 상수 ===
TACTIC_CLASSES = {
    '빌드업-크로스형': 0, '빌드업-중앙침투형': 1, '빌드업-중앙 침투형': 1,
    '빌드업-측면침투형': 2, '빌드업-측면 침투형': 2,
    '빌드업-롱볼형': 3, '빌드업-롱볼 형': 3,
    '세트피스-코너킥': 4, '세트피스-프리킥': 5,
    '세트피스-스로인': 6, '세트피스-골킥': 7,
    '수비-고위압박': 8, '수비-중위압박': 9, '수비-저위블록': 10,
    '공격전환': 11, '수비전환': 12,
}
NUM_TACTIC_CLASSES = 13
IDX_TO_TACTIC = {v: k for k, v in TACTIC_CLASSES.items()}

POSITION_NAMES = ['GK', 'DF', 'MF', 'FW', 'FP']
POS_TO_IDX = {n: i for i, n in enumerate(POSITION_NAMES)}
POS_TO_ONEHOT = {
    name: [1.0 if i == j else 0.0 for j in range(5)]
    for i, name in enumerate(POSITION_NAMES)
}

COMPLEMENTARITY_INIT = np.array([
    [0.1, 0.8, 0.4, 0.2, 0.5],
    [0.8, 0.6, 0.9, 0.5, 0.7],
    [0.4, 0.9, 0.5, 0.9, 0.7],
    [0.2, 0.5, 0.9, 0.3, 0.6],
    [0.5, 0.7, 0.7, 0.6, 0.5],
], dtype=np.float32)
COMPLEMENTARITY = COMPLEMENTARITY_INIT
POSITION_WEIGHTS = {'GK': 0.6, 'DF': 0.8, 'MF': 1.0, 'FW': 1.2, 'FP': 1.0}

NODE_IN_DIM = 11
EDGE_IN_DIM = 5
HIDDEN_DIM = 64
SAMPLE_RATE = 150   # 5초당 1프레임
MAX_MATCHES = 100   # 경기 수 제한 (메모리)

print(f'전술 {NUM_TACTIC_CLASSES}클래스, SAMPLE_RATE={SAMPLE_RATE}, MAX_MATCHES={MAX_MATCHES}')

## 2. 경기 메타데이터 로드 (Lazy — tag 데이터는 안 올림)

In [ ]:
def load_match_meta(match_dir: Path) -> dict | None:
    """메타데이터만 로드. track 프레임 데이터는 나중에 lazy load."""
    track_path = match_dir / '03' / 'track1.json'
    strategy_path = match_dir / '06' / 'strategy.json'

    if not track_path.exists() or not strategy_path.exists():
        return None

    # strategy.json (작은 파일)
    try:
        with open(strategy_path, encoding='utf-8-sig') as f:
            strategy = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return None

    # track1.json — playerInfo + 프레임 수만 추출 후 해제
    try:
        with open(track_path, encoding='utf-8-sig') as f:
            track = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return None

    fps = track.get('fps', 30)
    n_frames = len(track.get('tag', []))
    player_info_raw = track.get('playerInfo', [])
    del track  # 핵심: tag 데이터 메모리 해제
    gc.collect()

    # playerInfo 파싱
    player_map = {}
    for p in player_info_raw:
        sid = p.get('player_sid')
        if sid is None:
            continue
        pos = p.get('player_position', 'FP')
        if pos not in POSITION_NAMES:
            pos = 'FP'
        player_map[sid] = {'team': p.get('team', ''), 'position': pos}

    team_names = sorted(set(pm['team'] for pm in player_map.values() if pm['team'] != '심판'))
    if len(team_names) < 2:
        return None

    # 전술 라벨
    tactics = []
    for entry in strategy.get('strategyArray', []):
        off = entry.get('offense_strategy', {})
        d1 = off.get('depth1', '')
        tc = TACTIC_CLASSES.get(d1, -1)
        if tc >= 0:
            tactics.append({'start': entry['start'], 'end': entry['end'],
                            'tactic_class': tc, 'tactic_name': d1})

    if not tactics or n_frames == 0:
        return None

    return {
        'match_id': match_dir.name,
        'track_path': str(track_path),
        'fps': fps, 'n_frames': n_frames,
        'player_map': player_map,
        'team_to_id': {team_names[0]: 0, team_names[1]: 1},
        'tactics': tactics,
    }


match_dirs = sorted([d for d in AIHUB_DIR.iterdir() if d.is_dir()])
print(f'경기 폴더: {len(match_dirs)}개')

matches = []
for md in tqdm(match_dirs, desc='Loading meta'):
    m = load_match_meta(md)
    if m:
        matches.append(m)

print(f'유효 경기: {len(matches)}개')
print(f'총 프레임: {sum(m["n_frames"] for m in matches):,}개')
print(f'총 전술 라벨: {sum(len(m["tactics"]) for m in matches):,}개')

In [ ]:
# 전술 분포
tactic_dist = Counter()
for m in matches:
    for t in m['tactics']:
        tactic_dist[t['tactic_name']] += 1
print('=== 전술 분포 ===')
for name, count in tactic_dist.most_common():
    print(f'  {name:20s}: {count}')

# 포지션 분포
pos_dist = Counter()
for m in matches:
    for pm in m['player_map'].values():
        if pm['team'] != '심판':
            pos_dist[pm['position']] += 1
print(f'\n=== 포지션 분포 ===')
for pos, count in pos_dist.most_common():
    print(f'  {pos}: {count}')

## 3. 그래프 변환 함수

In [ ]:
class TacticalData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == 'link_pairs':
            return self.x.size(0)
        if key == 'prev_edge_index':
            return self.prev_x.size(0) if hasattr(self, 'prev_x') else 0
        return super().__inc__(key, value, *args, **kwargs)


def extract_frame(tag_entry, player_map, team_to_id,
                  img_w=1920.0, img_h=1080.0):
    """tag 엔트리 → {positions, team_ids, roles} or None."""
    positions, team_ids, roles = [], [], []
    for obj in tag_entry.get('data', []):
        sid = obj.get('id')
        bbox = obj.get('tag', [])
        if sid is None or len(bbox) < 4:
            continue
        info = player_map.get(sid)
        if info is None or info['team'] == '심판':
            continue
        team_id = team_to_id.get(info['team'])
        if team_id is None:
            continue
        cx = max(0.0, min(1.0, (bbox[0] + bbox[2] / 2) / img_w))
        cy = max(0.0, min(1.0, (bbox[1] + bbox[3] / 2) / img_h))
        positions.append([cx, cy])
        team_ids.append(team_id)
        roles.append(info['position'])
    if len(positions) < 10:
        return None
    return {
        'positions': np.array(positions, dtype=np.float32),
        'team_ids': np.array(team_ids, dtype=np.int64),
        'roles': roles,
    }


def get_tactic_for_frame(frame_num, tactics):
    for t in tactics:
        if t['start'] <= frame_num <= t['end']:
            return t['tactic_class']
    return -1


def frame_to_graph(positions, team_ids, prev_positions=None,
                   roles=None, k_neighbors=5):
    n = len(positions)
    k_neighbors = min(k_neighbors, n - 1)
    vel = (positions - prev_positions) if (prev_positions is not None and len(prev_positions) == n) else np.zeros_like(positions)
    bdist = np.full((n, 1), -1.0)
    pos_oh = np.array([POS_TO_ONEHOT.get(r, POS_TO_ONEHOT['FP']) for r in (roles or ['FP']*n)], dtype=np.float32)
    x = np.hstack([positions, vel, team_ids.reshape(-1, 1), bdist, pos_oh]).astype(np.float32)

    dm = cdist(positions, positions)
    src, dst, ef = [], [], []
    for i in range(n):
        for j in np.argsort(dm[i])[1:k_neighbors + 1]:
            src.append(i); dst.append(j)
            ef.append([dm[i, j], np.linalg.norm(vel[i] - vel[j]),
                       float(team_ids[i] == team_ids[j]),
                       positions[j, 0] - positions[i, 0],
                       positions[j, 1] - positions[i, 1]])
    return TacticalData(
        x=torch.tensor(x),
        edge_index=torch.tensor([src, dst], dtype=torch.long),
        edge_attr=torch.tensor(ef, dtype=torch.float32),
    )


def compute_synergy_targets(positions, team_ids, roles, n_samples=20, seed=0):
    n = len(positions)
    if roles is None: roles = ['FP'] * n
    dm = cdist(positions, positions)
    mx = dm.max() + 1e-8
    rng = np.random.default_rng(seed)
    same = [(a, b) for a in range(n) for b in range(a+1, n) if team_ids[a] == team_ids[b]]
    diff = [(a, b) for a in range(n) for b in range(a+1, n) if team_ids[a] != team_ids[b]]
    k = min(len(same), len(diff), n_samples)
    if k == 0:
        return torch.zeros((0, 2), dtype=torch.long), torch.zeros(0)
    pairs, labels = [], []
    for a, b in [same[i] for i in rng.choice(len(same), k, replace=False)]:
        comp = COMPLEMENTARITY[POS_TO_IDX.get(roles[a], 4), POS_TO_IDX.get(roles[b], 4)]
        pairs.append([a, b]); labels.append(float(comp * max(0, 1.0 - dm[a, b] / mx)))
    for a, b in [diff[i] for i in rng.choice(len(diff), k, replace=False)]:
        pairs.append([a, b]); labels.append(0.0)
    return torch.tensor(pairs, dtype=torch.long), torch.tensor(labels, dtype=torch.float32)


print('함수 준비 완료')

## 4. 그래프 데이터셋 구축 (Lazy Load — 경기별 로드 → 처리 → 해제)

In [ ]:
all_graphs = []
stats = {'valid': 0, 'labeled': 0, 'unlabeled': 0, 'skipped_matches': 0}

use_matches = matches[:MAX_MATCHES]
print(f'처리할 경기: {len(use_matches)}개 (MAX_MATCHES={MAX_MATCHES})\n')

for mi, match in enumerate(tqdm(use_matches, desc='Building graphs')):
    # 이 경기의 track1.json을 지금 로드
    try:
        with open(match['track_path'], encoding='utf-8-sig') as f:
            tags = json.load(f).get('tag', [])
    except Exception:
        stats['skipped_matches'] += 1
        continue

    pmap = match['player_map']
    tid_map = match['team_to_id']
    tactics = match['tactics']
    prev_graph = None
    prev_positions = None

    for idx in range(0, len(tags), SAMPLE_RATE):
        tag_entry = tags[idx]
        frame_num = int(tag_entry.get('frame', idx))

        frame = extract_frame(tag_entry, pmap, tid_map)
        if frame is None:
            prev_positions = None
            prev_graph = None
            continue

        stats['valid'] += 1
        positions = frame['positions']
        team_ids = frame['team_ids']
        roles = frame['roles']

        g = frame_to_graph(positions, team_ids, prev_positions, roles)

        # 전술 라벨
        tc = get_tactic_for_frame(frame_num, tactics)
        if tc >= 0:
            g.y = torch.tensor([tc], dtype=torch.long)
            g.y_mask = torch.tensor([1.0])
            stats['labeled'] += 1
        else:
            g.y = torch.tensor([0], dtype=torch.long)
            g.y_mask = torch.tensor([0.0])
            stats['unlabeled'] += 1

        g.change_label = torch.tensor([0.0])

        # 시너지 타겟
        lp, ll = compute_synergy_targets(positions, team_ids, roles, seed=stats['valid'])
        g.link_pairs = lp
        g.link_labels = ll

        # temporal
        if prev_graph is not None:
            g.prev_x = prev_graph.x.clone()
            g.prev_edge_index = prev_graph.edge_index.clone()
            g.prev_edge_attr = prev_graph.edge_attr.clone()
            g.prev_num_nodes = torch.tensor([prev_graph.num_nodes])
            g.has_prev = torch.tensor([1.0])
        else:
            g.prev_x = g.x.clone()
            g.prev_edge_index = g.edge_index.clone()
            g.prev_edge_attr = g.edge_attr.clone()
            g.prev_num_nodes = torch.tensor([g.num_nodes])
            g.has_prev = torch.tensor([0.0])

        prev_graph = g
        prev_positions = positions
        all_graphs.append(g)

    # 이 경기 프레임 데이터 해제
    del tags
    gc.collect()

# 변화점 라벨
ch = 0
for i in range(1, len(all_graphs)):
    if (all_graphs[i].y_mask.item() > 0 and all_graphs[i-1].y_mask.item() > 0
            and all_graphs[i].y.item() != all_graphs[i-1].y.item()):
        all_graphs[i].change_label = torch.tensor([1.0])
        ch += 1

print(f'\n=== 데이터셋 ===')
print(f'  그래프: {len(all_graphs)}개')
print(f'  라벨 있음: {stats["labeled"]} / 마스킹: {stats["unlabeled"]}')
print(f'  변화점: {ch}개')
print(f'  스킵 경기: {stats["skipped_matches"]}')

In [ ]:
# Train / Val / Test
train_g, test_g = train_test_split(all_graphs, test_size=0.15, random_state=42)
train_g, val_g = train_test_split(train_g, test_size=0.12, random_state=42)
print(f'Train: {len(train_g)}, Val: {len(val_g)}, Test: {len(test_g)}')

BS = 32
train_loader = DataLoader(train_g, batch_size=BS, shuffle=True)
val_loader = DataLoader(val_g, batch_size=BS)
test_loader = DataLoader(test_g, batch_size=BS)

## 5. 모델

In [ ]:
class SharedGATBackbone(nn.Module):
    def __init__(self, in_dim, hidden_dim, heads=4, edge_dim=5):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(in_dim)
        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.res_proj = nn.Linear(in_dim, hidden_dim)
    def forward(self, x, edge_index, edge_attr, batch):
        x0 = self.input_norm(x)
        h = F.elu(self.norm1(self.conv1(x0, edge_index, edge_attr)))
        h = F.dropout(h, p=0.1, training=self.training)
        h = F.elu(self.norm2(self.conv2(h, edge_index, edge_attr)))
        node_emb = h + self.res_proj(x0)
        return node_emb, torch.cat([global_mean_pool(node_emb, batch), global_max_pool(node_emb, batch)], dim=-1)

class TaskAdapter(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.LayerNorm(out_dim))
    def forward(self, x): return self.net(x)

class LearnableComplementarity(nn.Module):
    def __init__(self, init_matrix):
        super().__init__()
        init_t = torch.tensor(init_matrix, dtype=torch.float32).clamp(0.01, 0.99)
        self.C_logits = nn.Parameter(torch.log(init_t / (1.0 - init_t)))
    def forward(self, pi, pj, dist):
        return torch.sigmoid(self.C_logits)[pi, pj] * torch.exp(-dist)
    def get_matrix(self):
        with torch.no_grad(): return torch.sigmoid(self.C_logits).cpu().numpy()

class LinkPredictionHead(nn.Module):
    def __init__(self, emb_dim, init_matrix):
        super().__init__()
        self.complementarity = LearnableComplementarity(init_matrix)
        self.adapter = TaskAdapter(emb_dim, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*2, emb_dim), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(emb_dim, emb_dim//2), nn.GELU(), nn.Linear(emb_dim//2, 1))
    def forward(self, node_emb, pairs, pos_indices=None, distances=None):
        if pairs.numel() == 0: return torch.zeros(0, device=node_emb.device)
        adapted = self.adapter(node_emb)
        emb_score = torch.sigmoid(self.mlp(torch.cat([adapted[pairs[:,0]], adapted[pairs[:,1]]], dim=-1))).squeeze(-1)
        if pos_indices is not None and distances is not None:
            comp_score = self.complementarity(pos_indices[pairs[:,0]], pos_indices[pairs[:,1]], distances)
            return 0.5 * emb_score + 0.5 * comp_score
        return emb_score

class TacticClassificationHead(nn.Module):
    def __init__(self, graph_dim, num_classes):
        super().__init__()
        self.adapter = TaskAdapter(graph_dim, graph_dim//2)
        self.mlp = nn.Sequential(nn.Linear(graph_dim//2, graph_dim//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(graph_dim//2, num_classes))
    def forward(self, g): return self.mlp(self.adapter(g))

class ChangeDetectionHead(nn.Module):
    def __init__(self, graph_dim):
        super().__init__()
        self.adapter = TaskAdapter(graph_dim*3, graph_dim)
        self.mlp = nn.Sequential(nn.Linear(graph_dim, graph_dim//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(graph_dim//2, 1))
    def forward(self, t, p):
        return self.mlp(self.adapter(torch.cat([t, p, t - p], dim=-1)))

class UncertaintyWeighting(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n))
    def forward(self, *losses):
        total = torch.tensor(0.0, device=self.log_vars.device)
        for i, l in enumerate(losses):
            total = total + torch.exp(-self.log_vars[i]) * l + self.log_vars[i]
        return total

class TacticalGNN(nn.Module):
    def __init__(self, node_in_dim=11, hidden_dim=64, num_classes=13, gat_heads=4, edge_dim=5, comp_init=None):
        super().__init__()
        gd = hidden_dim * 2
        self.backbone = SharedGATBackbone(node_in_dim, hidden_dim, gat_heads, edge_dim)
        self.head_link = LinkPredictionHead(hidden_dim, comp_init if comp_init is not None else COMPLEMENTARITY_INIT)
        self.head_tactic = TacticClassificationHead(gd, num_classes)
        self.head_change = ChangeDetectionHead(gd)
        self.uncertainty = UncertaintyWeighting(3)
    def _pb(self, pnn):
        p = pnn.view(-1)
        return torch.repeat_interleave(torch.arange(p.size(0), device=p.device), p)
    def forward(self, data):
        ne, ge = self.backbone(data.x, data.edge_index, data.edge_attr, data.batch)
        _, pe = self.backbone(data.prev_x, data.prev_edge_index, data.prev_edge_attr, self._pb(data.prev_num_nodes))
        tl = self.head_tactic(ge)
        cl = self.head_change(ge, pe)
        ls = torch.zeros(0, device=ne.device)
        if hasattr(data, 'link_pairs') and data.link_pairs.numel() > 0:
            pi = data.x[:, 6:11].argmax(dim=1)
            pr = data.link_pairs
            ds = torch.norm(data.x[pr[:,0], :2] - data.x[pr[:,1], :2], dim=1)
            ls = self.head_link(ne, pr, pi, ds)
        return {'node_emb': ne, 'graph_emb': ge, 'tactic_logits': tl, 'change_logits': cl, 'link_scores': ls}

model = TacticalGNN(
    node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM,
    num_classes=NUM_TACTIC_CLASSES, edge_dim=EDGE_IN_DIM,
    comp_init=COMPLEMENTARITY_INIT,
).to(device)
print(f'TacticalGNN: {sum(p.numel() for p in model.parameters()):,} params')
print(f'초기 상보성 행렬:')
print(model.head_link.complementarity.get_matrix().round(2))

## 6. 학습

In [ ]:
cr = sum(g.change_label.item() for g in all_graphs) / max(len(all_graphs), 1)
cpw = max((1.0 - cr) / max(cr, 0.01), 1.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)
crit_t = nn.CrossEntropyLoss(reduction='none')
crit_l = nn.MSELoss()
crit_c = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cpw], device=device), reduction='none')

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    s = {'loss': 0, 'tc': 0, 'tn': 0, 'lm': 0, 'ln': 0}
    for batch in loader:
        batch = batch.to(device)
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            o = model(batch)
            y = batch.y.view(-1); mb = batch.y_mask.view(-1)
            cl = batch.change_label.view(-1); mc = batch.has_prev.view(-1)
            lt = (crit_t(o['tactic_logits'], y) * mb).sum() / mb.sum().clamp(min=1)
            lc = (crit_c(o['change_logits'].view(-1), cl) * mc).sum() / mc.sum().clamp(min=1)
            ll = torch.tensor(0.0, device=device)
            if o['link_scores'].numel() > 0:
                ll = crit_l(o['link_scores'], batch.link_labels.to(device))
                s['lm'] += ll.item() * len(o['link_scores'])
                s['ln'] += len(o['link_scores'])
            loss = model.uncertainty(lt, ll, lc)
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        s['loss'] += loss.item() * batch.num_graphs
        v = mb > 0
        if v.any():
            s['tc'] += (o['tactic_logits'][v].argmax(1) == y[v]).sum().item()
            s['tn'] += v.sum().item()
    return s['loss']/max(len(loader.dataset),1), s['tc']/max(s['tn'],1), s['lm']/max(s['ln'],1)

print(f'변화점 비율: {cr:.3f}, pos_weight: {cpw:.1f}')

In [ ]:
EPOCHS = 60
best_val_acc = 0.0
history = {k: [] for k in ['tl','vl','tt','vt','tlk','vlk']}

print('=== Training (Real Data) ===\n')
for epoch in range(1, EPOCHS + 1):
    tl, tt, tlk = run_epoch(train_loader, train=True)
    vl, vt, vlk = run_epoch(val_loader, train=False)
    scheduler.step()
    history['tl'].append(tl); history['vl'].append(vl)
    history['tt'].append(tt); history['vt'].append(vt)
    history['tlk'].append(tlk); history['vlk'].append(vlk)
    if vt > best_val_acc:
        best_val_acc = vt
        torch.save(model.state_dict(), str(MODEL_DIR / 'best_real.pt'))
    if epoch % 5 == 0 or epoch == 1:
        uw = torch.exp(-model.uncertainty.log_vars).detach().cpu().numpy()
        print(f'Epoch {epoch:3d} | Loss {tl:.4f}/{vl:.4f} | Tac {tt:.3f}/{vt:.3f} | '
              f'Link {tlk:.4f}/{vlk:.4f} | UW [{uw[0]:.2f},{uw[1]:.2f},{uw[2]:.2f}] | Best {best_val_acc:.3f}')

print(f'\nComplete. Best Val Acc: {best_val_acc:.3f}')

## 7. 평가

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['tl'], label='Train'); axes[0].plot(history['vl'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['tt'], label='Train'); axes[1].plot(history['vt'], label='Val')
axes[1].set_title('Tactic Acc'); axes[1].legend()
axes[2].plot(history['tlk'], label='Train'); axes[2].plot(history['vlk'], label='Val')
axes[2].set_title('Link MSE'); axes[2].legend()
plt.tight_layout(); plt.savefig(str(PROJECT_DIR / 'curves_real.png'), dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(str(MODEL_DIR / 'best_real.pt')))
tl, ta, tlk = run_epoch(test_loader, train=False)
print(f'=== Test (Real Data) ===')
print(f'Loss: {tl:.4f}, Tactic Acc: {ta:.3f}, Link MSE: {tlk:.4f}')

preds, labels = [], []
model.eval()
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        o = model(b)
        m = b.y_mask.view(-1) > 0
        if m.any():
            preds.extend(o['tactic_logits'][m].argmax(1).cpu().numpy())
            labels.extend(b.y.view(-1)[m].cpu().numpy())

pc = sorted(set(labels) | set(preds))
print('\n=== Tactic Classification (Real Data) ===')
print(classification_report(labels, preds, labels=pc,
      target_names=[IDX_TO_TACTIC.get(i, f'c{i}') for i in pc], zero_division=0))

## 8. 상보성 행렬 비교 + 저장

In [ ]:
learned = model.head_link.complementarity.get_matrix()
uw = torch.exp(-model.uncertainty.log_vars).detach().cpu().numpy()

ckpt = {
    'model_state_dict': model.state_dict(),
    'config': {'node_in_dim': NODE_IN_DIM, 'edge_dim': EDGE_IN_DIM,
               'hidden_dim': HIDDEN_DIM, 'num_classes': NUM_TACTIC_CLASSES},
    'complementarity_init': COMPLEMENTARITY_INIT.tolist(),
    'complementarity_learned': learned.tolist(),
    'task_weights': {'tactic': float(uw[0]), 'link': float(uw[1]), 'change': float(uw[2])},
    'history': history, 'best_val_acc': best_val_acc,
    'data': f'AI Hub track1.json, {len(use_matches)} matches, {len(all_graphs)} graphs',
}
torch.save(ckpt, str(MODEL_DIR / 'tactical_gnn_real.pt'))

print('=== 상보성 행렬 (도메인 → 실제 데이터) ===')
print(f'포지션: {POSITION_NAMES}\n')
print('초기값:'); print(COMPLEMENTARITY_INIT.round(2))
print(f'\n학습 후:'); print(learned.round(3))
print(f'\n차이:'); print(np.array2string(learned - COMPLEMENTARITY_INIT, precision=3, sign='+'))

# 히트맵
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mat, title in [(axes[0], COMPLEMENTARITY_INIT, 'Initial'),
                        (axes[1], learned, 'Learned'),
                        (axes[2], learned - COMPLEMENTARITY_INIT, 'Difference')]:
    vmin = -0.3 if 'Diff' in title else 0
    vmax = 0.3 if 'Diff' in title else 1
    im = ax.imshow(mat, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(5)); ax.set_xticklabels(POSITION_NAMES)
    ax.set_yticks(range(5)); ax.set_yticklabels(POSITION_NAMES)
    ax.set_title(title)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / 'complementarity_real.png'), dpi=150)
plt.show()

print(f'\n모델 저장: {MODEL_DIR / "tactical_gnn_real.pt"}')
print(f'Task weights: tactic={uw[0]:.3f}, link={uw[1]:.3f}, change={uw[2]:.3f}')